<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/08_시맨틱_검색과_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 8 서론](#chapter8)
* [Chapter 8-1 시맨틱 검색과 RAG 소개](#chapter8-1)
* [Chapter 8-2 언어 모델을 사용한 시맨틱 검색](#chapter8-2)
    * [Chapter 8-2-1 밀집 검색](#chapter8-2-1)    
    * [Chapter 8-2-2 리랭킹](#chapter8-2-2)    

## Chapter 8 서론 <a class="anchor" id="chapter8"></a>
1. LLM이 대중화되면서 사용자의 질문에 대한 답변에 잘못된 정보를 제공하는 '환각' 문제가 발생하고있다.

2. '환각'을 줄이는 대표적인 방법은 관련 정보를 검색하고 이를 LLM에게 전달하여 답변을 생성하는 RAG(Retrieval Augmented Generation) 방법이 있다.

## Chapter 8-1 시맨틱 검색과 RAG 소개 <a class="anchor" id="chapter8-1"></a>
1. 언어 모델을 검색에 활용하는 방법은 크게 세 개의 큰 범주로 나누어 진다.

2. 밀집 검색
    - 임베딩 개념을 사용하여 쿼리와 문서를 임베딩으로 변환 후 최근접 이웃을 찾는다.

3. 리랭킹
    - 쿼리와 결과의 관련성을 점수화 한 수 점수에 따라 결과를 재정렬한다.

4. RAG
    - 검색 기능을 통합하여 환각을 줄이고 더 정확한 답변을 생성하기 위해 생성 모델을 특정 데이터셋에 접목한 텍스트 생성 시스템이다.
    - 질문에 대한 답변을 생성하고 가능하면 정보의 출처를 인용한다.

## Chapter 8-2 언어 모델을 사용한 시맨틱 검색 <a class="anchor" id="chapter8-2"></a>
### Chapter 8-2-1 밀집 검색 <a class="anchor" id="chapter8-2-1"></a>
1. 임베딩은 텍스트를 벡터로 변환한다. 이를 아래의 그림과 같이 공간위에 놓인 한 점으로 생각할 수 있다.
   - 서로 가까이에 있는 포인트는 의미적으로 유사한 텍스트를 나타낸다.

2. 이 성질을 이용하여 사용자가 입력한 검색 쿼리를 임베딩 한후, 텍스트 아카이브와 동일한 공간에 투영한다.(동일한 차원의 벡터를 생성한다.)
    - 이 공간상에서 쿼리에 가장 가까운 문서가 결과가 된다.

        ![밀집 검색](./image/07_dense_search.png)

3. 외부 지식 데이터를 벡터 데이터베이스로 변환한 다음 이 벡터 데이터페이스에 쿼리하여 지식 정보를 검색한다.
    
    ![밀집 검색](./image/07_dense_search_2.png)

4. 밀집 검색 예제
    - 코희어를 사용해 위키백과에 있는 '인터스텔라' 영화 페이지에 담긴 내용을 검색한다.

In [16]:
# cohere를 사용하기위한 라이브러리를 임포트한다.
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm

# cohere API 키를 설정한다.
api_key = "m6E2WNLY0botDSYV5i9u2F9f26bw1TU7M3gsxG7G"

# cohere 클라이언트를 초기화한다.
co = cohere.Client(api_key)

In [18]:
# 위키백과의 인터스텔라 영화 문서에 있는 첫 번째 색션을 가져와, 텍스트 문장으로 나눈다.
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# 문장을 나누어 리스트로 만듭니다.
texts = text.split('.')

# 공백과 줄바꿈 문자를 삭제합니다.
texts = [t.strip(' \n') for t in texts]

texts

['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
 'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
 'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
 'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
 'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
 'Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm',
 'Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles',
 'Interstellar uses extensive practical a

In [21]:
# 문장 임베딩
# 임베딩을 만듭니다.
response = co.embed(
  texts=texts,
  input_type="search_document",
  model="embed-english-v3.0",
).embeddings

embeds = np.array(response)
print(embeds.shape)

(15, 1024)


In [23]:
# 검색 인덱스 구축하기
#   - 검색 인텍스는 임베딩을 저정하며 많은 데이터 포인트에서도 빠르게 최근접 이웃을 검색할 수 있도록 하는 데이터 구조입니다.
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim) # IndexFlatL2: L2 거리(유클리드 거리)를 사용하여 벡터 간의 거리를 계산하는 인덱스입니다. 
index.add(np.float32(embeds)) # 32비트 부동소수점을 실수를 저장하는 데이터 유형입니다. FAISS는 고성능 검색을 위해 32비트 부동소수점 형식을 사용합니다.

In [24]:
# 인덱스 검색하기

def search(query, number_of_results=3):
    # 쿼리를 임베딩한다.
    query_embed = co.embed(
        texts=[query],
        input_type="search_query",
        model="embed-english-v3.0",
    ).embeddings[0]

   # 최근접 이웃을 추출한다.
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

   # 데이터프레임을 사용해 출력을 준비한다.
    text_np = np.array(texts) # 인덱스 파싱을 쉽게하기 위해 텍스트 리스트를 넘파이 배열로 변환한다.
    results = pd.DataFrame(data={'텍스트': text_np[similar_item_ids[0]], '거리': distances[0]})

    # 결과를 출력하고 반환한다.
    print(f"쿼리: {query}\n 최근접 이웃: ")
    return results 

In [25]:
query = "how precise was the science"
results = search(query)
results

쿼리: how precise was the science
 최근접 이웃: 


,텍스트,거리
0,It has also received praise from many astronom...,1.203451
1,Caltech theoretical physicist and 2017 Nobel l...,1.370858
2,Interstellar uses extensive practical and mini...,1.582947


5. 최상위 결과는 질문에 대한 정확한 답변이지만, 결과에는 질문에 있는 키워드들이 포함되어 있지않다.
    - 이는 모델이 단어의 의미를 이해하고 유사한 의미를 가진 단어들을 연결할 수 있음을 보여준다.

In [27]:
# 상위 2개의 결과를 비교하기 위해 키워드 검색 함수를 만든다.
#   - BM25 알고리즘을 사용하여 키워드 검색을 수행하는 함수를 만든다.
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
   tokenized_doc = []
   for token in text.lower().split():
         token = token.strip(string.punctuation)
         if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
   return tokenized_doc

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

def  keyword_search(query, top_k=3, num_candidates=15):
    print(f"쿼리: {query}")

    # BM25 검색
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"BM25 검색 결과 (상위 {top_k}):")
    for hit in bm25_hits[:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace('\n', ' ')))
          

100%|██████████| 15/15 [00:00<00:00, 73498.32it/s]


In [28]:
keyword_search(query="how precise was the science")

쿼리: how precise was the science
BM25 검색 결과 (상위 3):
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


6. BM25 최상위 결과에는 키워드가 포함되어 있지만 질문에 대한 답은 아니다.
    - 리랭커를 사용해 BM25 검색 결과를 재정렬하여 질문에 대한 답변이 상위에 오도록 할 수 있다.

7. 사용자가 특정 구절과 정확하게 일치하는 결과가 필요할 때는 키워드 검색과 시맨틱 검색의 하이브리드 조합을 사용한다.

8. 텍스트를 청그로 나누기
    - 트랜스포머 언어 모델은 모델이 지원하는 토큰 수보더 더 긴 텍스트를 주입할 수 없다.
    - 문서를 청크로 나누어 각 청크에 임베딩을 생성한다. 검색 시 쿼리를 임베딩하여 가장 유사한 청크를 검색한다.

9. 문서당 여러개의 벡터로 임베딩
    - 문서를 더 작안 단위인 청크로 나누고 이런 청크를 임베딩한다.
    - 검색 인텍스는 청크 임베딩의 인덱스가 된다.
    - 텍스트 전체를 포괄하고 벡터가 텍스트 안에 있는 개별 개념을 포착하기 때문에 더 낫다.

10. 일반적인 청크 분할 전략은 다음과 같다.
    - 문서 제목을 청크에 추가한다.
    - 청크 앞뒤에 있는 일부 텍스트를 추가하여, 주변 텍스트가 인접한 청크에 나타나토록 한다.

        ![청크 분할 전략](./image/07_chunking_strategy.png)

11. 최근접 이웃 검색 vs 벡터 테이터 베이스
    - 수백만 개의 벡터에서 최근접 이웃을 검색하는 최적의 방법은 Annoy나 FAISS와 같은 라이브러리를 사용하는 것이다.
    - Weaviate나 Pinecone과 같은 벡터 데이터 베이스를 사용하면 인덱스를 재구축 할 필욥 없이 벡터를 추가, 삭제, 업데이트 할 수 있다.

12. 밀집 검색을 위해 임베딩 모델 미세 튜닝하기
    - 검색도 단순한 토큰 임베딩이 아니라 텍스트 임베딩을 최적화 할 필요가 있다.
    - 미세 튜닝 과정은 쿼리의 임베딩이 결과 문장의 임베뎅에 가깝도록 모델을 훈련하는 것이다.
    - 문장과 관련이 없는 네거티브 쿼리 샘플로 모델이 관련이 없는 텍스트를 멀리 떨어뜨리도록 훈련한다.

### Chapter 8-2-2 리랭킹 <a class="anchor" id="chapter8-2-2"></a>
